In [30]:
from pathlib import Path
import jsonlines
from xopen import xopen
import orjson

In [31]:
IEPATH = "/Data_two/wyw/data/CETC_PRODUCT/task1/enwiki/*"
RAWPATH = "/Data_two/wyw/data/CETC_PRODUCT/raw/enwiki/*.zst"
OUTPATH = "/Data_two/wyw/data/CETC_PRODUCT/merge/enwiki_task1.jsonl"

In [32]:
raw_paths = list(Path(RAWPATH).parent.glob(Path(RAWPATH).name))

In [33]:
zh_input = dict()
need_cols = ["page_id", "title", "abstract", "entity_classification", "domain_and_region_classifier"]
for raw_path in raw_paths:
    with xopen(raw_path, 'rt') as f:
        for line in f:
            try:
                obj = orjson.loads(line)
            except:
                continue
            zh_input[obj['page_id']] = {col: obj.get(col, "") for col in need_cols}  

In [34]:
len(zh_input)

2614231

In [35]:
def is_need(record:dict)->bool:
    if record["out"]["error"]:
        return False
    if record["out"]["parse"]["consolidated_view"]:
        return True
    return False

def process_id(_id:list|str)->tuple:
    if isinstance(_id, str):
        split_res = _id.split("_", maxsplit=1)
    else:
        split_res = _id
    
    return {
        "page_id": split_res[0],
        "title": split_res[1]
    }
def process_parse(record:dict)->tuple:
    new_record = dict()
    for k, v in record.items():
        if k in ["entity_classification", "domain_and_region_classifier"]:
            new_record[k] = v["result"]
        
        else:
            new_record[k] = v
    return new_record

In [36]:
iepaths = list(Path(IEPATH).parent.glob(Path(IEPATH).name))
iepaths = sorted(iepaths,key = lambda x:x.name, reverse = True)
writed_ids = set()
results = []
with jsonlines.open(OUTPATH, mode='w') as writer:
    for ie_path in iepaths:
        with jsonlines.open(ie_path) as reader:
            for obj in reader.iter(skip_invalid=True, skip_empty=True):

                id_dict= process_id(obj["id"])
                id_tuple = tuple(id_dict.values())
                if not is_need(obj) or id_tuple in writed_ids:
                    continue
                out_obj = process_parse(zh_input[id_dict["page_id"]]) | obj["out"]["parse"]
                writer.write(out_obj)
                results.append(out_obj)
                writed_ids.add(id_tuple)

In [20]:
iepaths

[PosixPath('/Data_two/wyw/data/CETC_PRODUCT/task1/zhwiki/zhwiki_20251126_093439'),
 PosixPath('/Data_two/wyw/data/CETC_PRODUCT/task1/zhwiki/zhwiki_20251116_133309'),
 PosixPath('/Data_two/wyw/data/CETC_PRODUCT/task1/zhwiki/20251126_173407'),
 PosixPath('/Data_two/wyw/data/CETC_PRODUCT/task1/zhwiki/20251126_171052'),
 PosixPath('/Data_two/wyw/data/CETC_PRODUCT/task1/zhwiki/20251126_162337'),
 PosixPath('/Data_two/wyw/data/CETC_PRODUCT/task1/zhwiki/20251126_162044'),
 PosixPath('/Data_two/wyw/data/CETC_PRODUCT/task1/zhwiki/20251126_161546'),
 PosixPath('/Data_two/wyw/data/CETC_PRODUCT/task1/zhwiki/20251126_154228'),
 PosixPath('/Data_two/wyw/data/CETC_PRODUCT/task1/zhwiki/20251126_111314'),
 PosixPath('/Data_two/wyw/data/CETC_PRODUCT/task1/zhwiki/20251126_093439')]

In [28]:
len(results)

71034

In [25]:
out_obj

{'page_id': '7736041',
 'title': 'San Joaquin and Sierra Nevada Railroad',
 'abstract': 'The San Joaquin and Sierra Nevada Railroad (or Rail Road) was originally built as a narrow gauge that ran from Bracks Landing (10.6 miles west of Woodbridge on the San Joaquin Delta, on the Brack Tract on the east side of South Mokelumne River, between Hog Slough and Terminous) to Woodbridge and Lodi and then east to the Sierra Nevada foothill town of Valley Springs. The railroad was incorporated on March 28, 1882 and construction was completed on April 15, 1885. The railroad was built as a comm...',
 'entity_classification': {'type': '组织',
  'subtype': '公司企业',
  'rationale': 'San Joaquin and Sierra Nevada Railroad 是一个具体的铁路公司，属于组织类别中的公司企业子类别。'},
 'domain_and_region_classifier': {'domains': ['工程与技术'],
  'regions': ['美国'],
  'rationale': '该实体是铁路公司，涉及铁路建设，属于工程与技术领域；其总部和主要运营地点在加利福尼亚州，属于美国地区。'},
 'consolidated_view': {'alias': ['Rail Road'],
  'persons': None,
  'locations': [{'country': 'United States'

In [24]:
OUTPATH

'/Data_two/wyw/data/CETC_PRODUCT/merge/enwiki_task1.jsonl'

In [17]:
tuple(id_dict.values())

('61768725', 'Beresford Parlett')